# Get to Know a Dataset: UChicago AI Weather Model Reforecast Archive

This notebook serves as a guided tour of the [UChicago AI Weather Model Reforecast Archive](https://registry.opendata.aws/uchicago-ai-weather-reforecast) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

The archive holds long-range reforecasts from several AI weather models (GraphCast, Aurora 1.5, AIFS-single v2, AIFS-ENS v2, FGN, Aurora 1.5 ENS, and NeuralGCM), initialized daily from 2000 to 2025 and run to 50 days under a common protocol. Full documentation, including every model's variables, units, and temporal aggregation, is at <https://envfluids.github.io/reforecast/>.

*These reforecasts are experimental machine-learning output produced for research use. They are not official products of any model provider or meteorological agency.*

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The bucket is organized by **model**, then by **initialization date**. Each forecast is a self-contained [Zarr](https://zarr.dev/) store:

```
s3://uchicago-ai-weather-reforecast/
    <model>/
        20000101T00.zarr/
        20000102T00.zarr/
        ...
        20251231T00.zarr/
```

- **One store is one forecast:** a single 00 UTC initialization run out to 50 days, including all 51 members for ensemble models.
- **Every model shares the same initialization dates,** so comparing models on the same forecast means changing only the model prefix.
- **New models are added as new prefixes,** run under the same protocol, so existing paths never change.

The full archive is **multi-petabyte**. The intended workflow is to open forecasts lazily and read only the variables, lead times, and regions you need.

We'll use [xarray](https://docs.xarray.dev/) to work with the data and [s3fs](https://s3fs.readthedocs.io/) to read from S3.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# xarray >= 2024.10.0
# zarr >= 2.18
# dask >= 2024.10.0
# s3fs >= 2024.10.0
# matplotlib >= 3.8

import numpy as np
import xarray as xr
import s3fs
import matplotlib.pyplot as plt

The bucket is public, so we connect anonymously; no AWS account is needed. Listing the top level shows one prefix per model.

In [ ]:
BUCKET = "uchicago-ai-weather-reforecast"
fs = s3fs.S3FileSystem(anon=True)

fs.ls(BUCKET)

Inside each model's prefix is one store per initialization date.

<!-- TODO (data provider): confirm these prefix names against the listing above. -->

In [ ]:
DET_MODEL = "graphcast"   # a deterministic model
ENS_MODEL = "aifs-ens"    # an ensemble model

stores = fs.ls(f"{BUCKET}/{DET_MODEL}")
print(f"{len(stores)} forecasts, from {stores[0].split('/')[-1]} to {stores[-1].split('/')[-1]}")

A Zarr store is itself a small directory tree: one prefix per variable or coordinate (holding its compressed chunks) plus metadata describing the whole store. Zarr-aware libraries handle this for you, but it's useful to know what's there.

In [ ]:
INIT = "20200101"
fs.ls(f"{BUCKET}/{DET_MODEL}/{INIT}T00.zarr")

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

All forecasts are stored in **[Zarr](https://zarr.dev/)**, a cloud-native format for chunked, compressed, N-dimensional arrays. Each array is split into chunks stored as separate objects, so a program can read one variable at one lead time over one region without downloading the rest. That's why we chose it: the archive is multi-petabyte, so almost nobody wants all of it, and Zarr lets you pay only for the slice you use.

**Tools.** `xarray.open_zarr` reads only metadata up front and loads values lazily (via [dask](https://www.dask.org/)) when you compute or plot. This is the easiest way to work with the archive in Python.

**What to know about this archive:**

- **Dimensions** are `(time, [number,] prediction_timedelta, lat, lon)`. Each store has a single `time` (the initialization), `number` indexes ensemble members, and `prediction_timedelta` is lead time.
- **There are two lead-time axes.** Fields used for extremes (`2t`, `mslp`, low-level `t` and `q`) are 6-hourly on `prediction_timedelta`. Circulation fields (`z_*`, `u_*`, `v_*`) are daily means and precipitation (`tp`) is a daily total, all on `prediction_timedelta_daily`. Days are UTC calendar days.
- **Naming exceptions:** mean sea level pressure is `msl` in AIFS-ENS v2 (`mslp` elsewhere), and NeuralGCM uses long names like `geopotential_500_avg` on a coarser ≈2.8° grid.
- **Grids** are 721 × 1440 (0.25°), except the Aurora models at 720 × 1440. Align them before differencing models.
- **Units** are SI: K, Pa, m (precipitation), and m² s⁻² (geopotential; divide by 9.80665 for height in meters). All data are float32.

**AWS services.** The bucket is in `us-west-2`. Running on [Amazon EC2](https://aws.amazon.com/ec2/) or [Amazon SageMaker](https://aws.amazon.com/sagemaker/) in that region gives the fastest reads, and a dask cluster lets you process many forecasts in parallel.

We'll open forecasts with one small helper. It drops the single-valued `time` dimension and converts lead times to days, which makes selecting and plotting simpler.

In [ ]:
def open_forecast(model, init):
    path = f"{BUCKET}/{model}/{init}T00.zarr"
    ds = xr.open_zarr(fs.get_mapper(path)).isel(time=0)
    return ds.assign_coords(   # lead times as days, e.g. 0.25, 0.5, ... 50
        prediction_timedelta=ds.prediction_timedelta / np.timedelta64(1, "D"),
        prediction_timedelta_daily=ds.prediction_timedelta_daily / np.timedelta64(1, "D"),
    )

The chunk shape tells you the smallest unit that gets read from S3, and therefore what selections are cheap.

<!-- TODO (data provider): once uploaded, add one sentence describing the chunk layout. -->

In [ ]:
ds = open_forecast(DET_MODEL, INIT)
print("dims:      ", ds["2t"].dims)
print("shape:     ", ds["2t"].shape)
print("chunk size:", ds["2t"].data.chunksize)

### Q: Can you show us an example of downloading and loading data from your dataset?

Opening a forecast is nearly instant because only metadata is read. The summary shows every variable and coordinate without loading any values.

In [ ]:
ds

The split between 6-hourly and daily variables is worth seeing at a glance:

In [ ]:
for lead_dim in ["prediction_timedelta", "prediction_timedelta_daily"]:
    print(f"{lead_dim}: {[v for v in ds.data_vars if lead_dim in ds[v].dims]}")

Now let's load some values: the 2 m temperature forecast at Chicago (41.88° N, 87.63° W) over all 50 days. Longitudes run 0–360, so 87.63° W is 272.37°. Selecting a point before calling `.load()` means only the chunks covering that point are read.

In [ ]:
chicago = ds["2t"].sel(lat=41.88, lon=272.37, method="nearest").load() - 273.15

chicago.plot(figsize=(10, 4))
plt.ylabel("2 m temperature (°C)")
plt.xlabel("Lead time (days)");

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

**Fifty-day rollouts.** Most AI weather models were built and evaluated for forecasts of 10–15 days. This archive runs them to 50. Here is one GraphCast forecast at four lead times. Seeing whether fields stay physically realistic, smooth out, or drift at long leads is one of the questions the archive was built to answer.

In [ ]:
days = [1, 10, 30, 50]
maps = ds["2t"].sel(prediction_timedelta=days, method="nearest") - 273.15

grid = maps.plot(col="prediction_timedelta", col_wrap=2, cmap="RdYlBu_r", vmin=-40, vmax=40,
                 figsize=(14, 7), cbar_kwargs={"label": "2 m temperature (°C)"})
for ax, day in zip(grid.axs.flat, days):
    ax.set_title(f"Day {day}")

**Ensemble spread.** Ensemble stores hold all 51 members. Plotting every member at one location shows how quickly forecasts diverge as predictability is lost.

In [ ]:
ens = open_forecast(ENS_MODEL, INIT)
members = ens["2t"].sel(lat=41.88, lon=272.37, method="nearest").load() - 273.15

fig, ax = plt.subplots(figsize=(10, 4))
members.plot.line(x="prediction_timedelta", hue="number", ax=ax, color="tab:blue", alpha=0.15, add_legend=False)
members.mean("number").plot(ax=ax, color="black", lw=2, label="ensemble mean")
ax.set(xlabel="Lead time (days)", ylabel="2 m temperature (°C)", title=f"{ENS_MODEL}, Chicago")
ax.legend();

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

<!-- TODO (data provider): replace or extend with a result from your own work if you have one. -->

**How fast does ensemble spread grow, and when does it level off?**

Spread measures how much ensemble members disagree. It grows as small differences amplify, then levels off once members are as different as unrelated weather states. The lead time where it saturates is a rough guide to how far ahead the model retains useful information, which is a natural question for models run well past their usual range.

We measure the spread of 500 hPa geopotential height over the Northern Hemisphere midlatitudes (30–70° N): the standard deviation across members at each grid point, averaged over the region with cosine-latitude weights so that the smaller high-latitude grid cells don't count extra.

In [ ]:
def midlatitude_spread(model, init):
    z500 = open_forecast(model, init)["z_500"] / 9.80665     # geopotential height (m)
    z500 = z500.sel(lat=slice(70, 30))                        # latitude runs north to south
    weights = np.cos(np.deg2rad(z500.lat))
    return z500.std("number").weighted(weights).mean(["lat", "lon"]).load()

spread = midlatitude_spread(ENS_MODEL, INIT)

spread.plot(marker="o", figsize=(10, 4))
plt.ylabel("Z500 ensemble spread (m)")
plt.xlabel("Lead time (days)");

One way to put a number on saturation is the first lead day at which spread reaches 95% of its final value.

In [ ]:
saturated = spread.where(spread >= 0.95 * spread[-1], drop=True)
print(f"Spread reaches 95% of its final value by day {saturated.prediction_timedelta_daily[0].item():.0f}.")

One forecast is noisy. A more robust answer averages over many initializations, for example the first of each month across several years, and since every model shares the same dates, the same loop can compare ensemble models side by side. Each forecast is read separately, so start with a few and scale up, ideally on EC2 in `us-west-2`.

In [ ]:
inits = ["20200101", "20200201", "20200301"]   # extend to more dates or years

mean_spread = xr.concat([midlatitude_spread(ENS_MODEL, d) for d in inits], dim="init").mean("init")

mean_spread.plot(figsize=(10, 4), label=f"{ENS_MODEL}, mean of {len(inits)} forecasts")
plt.ylabel("Z500 ensemble spread (m)")
plt.xlabel("Lead time (days)")
plt.legend();

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

<!-- TODO (data provider): edit to reflect the question you most want the community to take on. -->

**Can a blend of AI weather models beat every individual model at subseasonal lead times (weeks 3–6)?**

Skillful forecasts two to six weeks ahead would help agriculture, energy, water management, and heat-health planning, but this range has long been one of the hardest in weather prediction. Different AI models make different errors, so a combination may outperform any single one. This archive makes the question testable at scale: seven models, the same 26 years of daily initializations, and rollouts long enough to reach weeks 3–6.

Advice for anyone taking this on:

- **Verify against reanalysis** such as ERA5, and compare weekly averages rather than single days, since day-to-day weather is essentially unpredictable at these leads.
- **Start small:** one variable (weekly 2 m temperature or 500 hPa height), one region, and a few years.
- **Use proper scores,** such as the continuous ranked probability score (CRPS) for ensembles and anomaly correlation for deterministic forecasts.
- **Cross-validate by year,** so blending weights are never tuned on the forecasts they're scored on.
- **Align grids and names first** (721 vs. 720 latitudes, NeuralGCM's coarser grid and naming). Weighting models by past skill is a meaningful baseline.

We'd love to hear what you find. Share questions and results on [AWS re:Post with the AWS Open Data tag](https://repost.aws/tags/TApd0Wl5P8S9O6riTWmh-cGw/aws-open-data) or email marchakitus@uchicago.edu.